In [5]:

# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 2. Load Dataset

df = pd.read_csv("glass.csv")


# 3. Inspect Data

print("Shape of dataset:", df.shape)
print("Columns:", df.columns)
print(df.head())

# If there is an Id column, drop it
if "Id" in df.columns:
    df = df.drop(columns=["Id"])

# ==============================
# 4. Convert to Binary Problem
# Type 1 -> 1
# Others -> 0
# ==============================

df["y"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])


# ==============================
# 5. Separate Inputs and Labels
# ==============================

X = df.drop(columns=["y"]).values
y = df["y"].values


# ==============================
# 6. Train-Test Split
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ==============================
# 7. Feature Scaling
# ==============================

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ==============================
# 8. Sigmoid Function
# ==============================

def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# ==============================
# 9. Forward Pass (Probability)
# ==============================

def predict_proba(X, w, b):
    z = X @ w + b
    return sigmoid(z)


# ==============================
# 10. Loss Function (Binary Cross Entropy)
# ==============================

def loss(y, p):
    # small epsilon to avoid log(0)
    epsilon = 1e-9
    p = np.clip(p, epsilon, 1 - epsilon)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))


# ==============================
# 11. Update Weights
# ==============================

def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y

    w = w - lr * (X.T @ error) / len(y)
    b = b - lr * np.mean(error)

    return w, b


# ==============================
# 12. Training Loop
# ==============================

w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1
epochs = 100

for epoch in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)

    if epoch % 10 == 0:
        p_train = predict_proba(X_train, w, b)
        current_loss = loss(y_train, p_train)
        print(f"Epoch {epoch}, Loss: {current_loss:.4f}")


# ==============================
# 13. Prediction Function
# ==============================

def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)


# ==============================
# 14. Evaluate Model
# ==============================

# Probabilities
p_test = predict_proba(X_test, w, b)

# Predictions with threshold 0.5
y_pred_05 = predict_label(p_test, threshold=0.5)

# Predictions with threshold 0.7
y_pred_07 = predict_label(p_test, threshold=0.7)

# Accuracy calculation
accuracy_05 = np.mean(y_pred_05 == y_test)
accuracy_07 = np.mean(y_pred_07 == y_test)

print("\nFinal Weights:", w)
print("Final Bias:", b)

print("\nAccuracy (threshold = 0.5):", accuracy_05)
print("Accuracy (threshold = 0.7):", accuracy_07)

Shape of dataset: (214, 10)
Columns: Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')
        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1
Epoch 0, Loss: 0.6822
Epoch 10, Loss: 0.6107
Epoch 20, Loss: 0.5748
Epoch 30, Loss: 0.5529
Epoch 40, Loss: 0.5379
Epoch 50, Loss: 0.5269
Epoch 60, Loss: 0.5184
Epoch 70, Loss: 0.5116
Epoch 80, Loss: 0.5060
Epoch 90, Loss: 0.5014

Final Weights: [ 0.10796659 -0.15606355  0.63480916 -0.67204019  0.07051998 -0.08936205
 -0.19026934 -0.16364966 -0.1099809 ]
Final Bias: -0.7346250485229616

Accuracy (threshold = 0.5): 0.8604651162790697
Accuracy (threshold = 0.7): 0.7209302325581395
